# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Load & Inspect data

In [0]:
df_raw = spark.table("workspace.bronze.erp_loc_a101_raw")
df_raw.display()

In [0]:
print(f'There are {df_raw.count()} rows')
df_raw.printSchema()

In [0]:
id_count = df_raw.select('CID').distinct().count()
print(f'There are {id_count} unique ids')

In [0]:
df_raw.select('CNTRY').distinct().display()

# Transform data

## Rename columns

In [0]:
name_map = {
    "CID": "customer_key",
    "CNTRY": "country"
}

df = df_raw
for c in df.columns:
    df = df.withColumnRenamed(c, name_map[c])
    
df.display()

## Clean 'country'

In [0]:

country = F.trim(F.col("country"))
df = df.withColumn(
    "country",
    F.when(country == "", F.lit(None))
     .when(country.isin("United States", "USA"), F.lit("US"))
     .when(country == "United Kingdom", F.lit("UK"))
     .when(country == "DE", F.lit("Germany"))
     .otherwise(country)
)

df.groupBy('country').count().display()

## Align 'customer_key'

In [0]:
%python
df = df.withColumn(
    'customer_key',
    F.regexp_replace(F.col('customer_key'), "-", "")
)
df.display()

# Write table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customer_location")

## Sanity check

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location